**Theme:**

*“A model that performs well on training data but poorly on new data is useless.”*

# Imports

In [1]:
import torch 
import torch.nn as nn 

torch.__version__

'2.8.0+cu129'

In [4]:
from torchvision import datasets, transforms 
from torch.utils.data import DataLoader, random_split   

# Overfitting, Validation and Learning Curves

## Create a validation set

In [6]:
transform = transforms.ToTensor() 

train_data_full = datasets.MNIST(root='data', train=True, download=True, transform=transform)  

train_size = int(0.8 * len(train_data_full))   
val_size = len(train_data_full) - train_size 

train_data, val_data = random_split(train_data_full, [train_size, val_size])  

train_loader = DataLoader(train_data, batch_size=64, shuffle=True) 
val_loader = DataLoader(val_data, batch_size=64, shuffle=False) 

In [7]:
images, labels = next(iter(train_loader)) 
images.shape

torch.Size([64, 1, 28, 28])

In [13]:
# Neural Network
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128,10),
        )
    
    def forward(self, x):
        return self.net(x) 

model = MLP()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  

Watch:

- Train loss

- Validation loss

## Track train and validation loss

In [14]:
for epoch in range(20):
    # TRAIN 
    model.train()
    train_loss = 0.0 
    for images, labels in train_loader:
        # forward pass 
        outputs = model(images) 
        # compute loss 
        loss = loss_fn(outputs, labels) 

        # zero gradient 
        optimizer.zero_grad() 
        # backward pass
        loss.backward() 
        # update parameters 
        optimizer.step()

        train_loss += loss.item() 

    # VALIDATION 
    model.eval() 
    val_loss = 0.0 
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images) 
            loss = loss_fn(outputs, labels) 

            val_loss += loss.item() 

    print(f"Epoch: {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Validation Loss: {val_loss/len(val_loader):.4f}")

Epoch: 1 | Train Loss: 0.3741 | Validation Loss: 0.2186
Epoch: 2 | Train Loss: 0.1732 | Validation Loss: 0.1628
Epoch: 3 | Train Loss: 0.1234 | Validation Loss: 0.1319
Epoch: 4 | Train Loss: 0.0962 | Validation Loss: 0.1142
Epoch: 5 | Train Loss: 0.0759 | Validation Loss: 0.1089
Epoch: 6 | Train Loss: 0.0627 | Validation Loss: 0.1008
Epoch: 7 | Train Loss: 0.0518 | Validation Loss: 0.0874
Epoch: 8 | Train Loss: 0.0429 | Validation Loss: 0.0942
Epoch: 9 | Train Loss: 0.0352 | Validation Loss: 0.0857
Epoch: 10 | Train Loss: 0.0287 | Validation Loss: 0.0874
Epoch: 11 | Train Loss: 0.0250 | Validation Loss: 0.0878
Epoch: 12 | Train Loss: 0.0199 | Validation Loss: 0.0881
Epoch: 13 | Train Loss: 0.0176 | Validation Loss: 0.0901
Epoch: 14 | Train Loss: 0.0146 | Validation Loss: 0.0919
Epoch: 15 | Train Loss: 0.0113 | Validation Loss: 0.0864
Epoch: 16 | Train Loss: 0.0107 | Validation Loss: 0.0918
Epoch: 17 | Train Loss: 0.0085 | Validation Loss: 0.1004
Epoch: 18 | Train Loss: 0.0071 | Validat

- What happens when train loss keeps decreasing but val loss stops decreasing?

- What does that mean?

## Add dropout (Anti-Overfitting)

In [15]:
# Multilayer Perceptron Algorithm 
class MLP2(nn.Module):
    def __init__(self):
        super().__init__() 
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128),
            nn.ReLU(), 
            nn.Dropout(0.3), 
            nn.Linear(128, 10)
        )
    
    def forward(self, x):
        return self.net(x) 

model2 = MLP2()
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.001) 

In [16]:
for epoch in range(20):
    model2.train()
    train_loss = 0.0 

    for images, labels in train_loader: 
        outputs = model2(images) 
        loss = loss_fn(outputs, labels) 

        optimizer2.zero_grad() 
        loss.backward() 
        optimizer2.step() 

        train_loss += loss.item() 

    model2.eval()
    val_loss = 0.0 

    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model2(images) 
            loss = loss_fn(outputs,labels) 

            val_loss += loss.item() 

    print(f"Epoch: {epoch+1} | TrainLoss: {train_loss/len(train_loader):.3f} | ValLoss: {val_loss/len(val_loader):.3f}") 

Epoch: 1 | TrainLoss: 0.439 | ValLoss: 0.227
Epoch: 2 | TrainLoss: 0.218 | ValLoss: 0.164
Epoch: 3 | TrainLoss: 0.166 | ValLoss: 0.138
Epoch: 4 | TrainLoss: 0.141 | ValLoss: 0.118
Epoch: 5 | TrainLoss: 0.117 | ValLoss: 0.109
Epoch: 6 | TrainLoss: 0.107 | ValLoss: 0.102
Epoch: 7 | TrainLoss: 0.095 | ValLoss: 0.092
Epoch: 8 | TrainLoss: 0.087 | ValLoss: 0.096
Epoch: 9 | TrainLoss: 0.079 | ValLoss: 0.088
Epoch: 10 | TrainLoss: 0.073 | ValLoss: 0.087
Epoch: 11 | TrainLoss: 0.070 | ValLoss: 0.094
Epoch: 12 | TrainLoss: 0.065 | ValLoss: 0.082
Epoch: 13 | TrainLoss: 0.059 | ValLoss: 0.079
Epoch: 14 | TrainLoss: 0.055 | ValLoss: 0.080
Epoch: 15 | TrainLoss: 0.054 | ValLoss: 0.079
Epoch: 16 | TrainLoss: 0.050 | ValLoss: 0.087
Epoch: 17 | TrainLoss: 0.048 | ValLoss: 0.083
Epoch: 18 | TrainLoss: 0.050 | ValLoss: 0.084
Epoch: 19 | TrainLoss: 0.045 | ValLoss: 0.087
Epoch: 20 | TrainLoss: 0.042 | ValLoss: 0.084


- What changed?

- Why did dropout help?

# Ineraction

> Overfitting happens when a model becomes too complex and starts fitting noise instead of learning the true data pattern.

And:

> Dropout forces the network to not rely on any single neuron, making it learn more robust and general features.

### Table to know

| Training loss | Validation loss | Problem          |
| ------------- | --------------- | ---------------- |
| High          | High            | **Underfitting** |
| Low           | High            | **Overfitting**  |
| Low           | Low             | Good model       |


### Fix underfitting

**Underfitting = model too simple**

Use:

- Bigger model (more layers / neurons)

- Train longer

- Reduce regularization

- Better features

Goal: increase model capacity

---

### Fix overfitting



Overfitting = model too powerful

Use:

- Dropout

- L2 regularization (weight decay)

- Data augmentation

- Smaller model

- Early stopping

- More data

Goal: reduce model capacity

---

### Why dropout workes



By adding dropout:

- You weakened the model

- Forced it to generalize

- Prevented memorization

That’s how validation loss improves.

> Interview-level answer

- If training and validation loss are both high, the model is underfitting — increase capacity.
- If training loss is low but validation loss is high, the model is overfitting — add regularization.